# Day 22：Frontend Crash Course（HTML5 / CSS3 / JS fetch 静态对话台）

> **🎙️ 讲师旁白**：欢迎来到星辰数科《智枢企业级智能体平台》第二阶段第 22 天！今天我们彻底告别纯命令行交互，使用原生 HTML5 + CSS3 Flexbox + JavaScript `fetch()` 打造响应式中文 Web 对话台（`v0.2.0-day22`），并用 Python `ui_validator` 对前端交付物执行自动化结构校验。

## 1. 环境初始化与导入 `project_iterations/day_22` 模块
定位 `project_iterations/day_22` 目录并导入 `zhishu_agent.ui_validator` 工具函数。

In [1]:
import json
import sys
from pathlib import Path

repo_root = Path.cwd()
while not (repo_root / "project_iterations").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
day22_dir = repo_root / "project_iterations" / "day_22"
if str(day22_dir) not in sys.path:
    sys.path.insert(0, str(day22_dir))

from zhishu_agent import (
    REQUIRED_DOM_IDS,
    describe_static_serving,
    get_frontend_dir,
    get_index_html_path,
    validate_frontend_file,
    validate_html_content,
)

print("静态目录:", get_frontend_dir())
print("入口页面:", get_index_html_path())
print("必需 DOM ID:", REQUIRED_DOM_IDS)

静态目录: /home/antigravity/ab6539763/-AI-Agent/project_iterations/day_22/frontend
入口页面: /home/antigravity/ab6539763/-AI-Agent/project_iterations/day_22/frontend/index.html
必需 DOM ID: ('chat-messages', 'chat-form', 'user-input', 'send-btn')


## 2. 自动化校验 `frontend/index.html` 结构与核心规范
使用 `validate_frontend_file()` 解析 HTML DOM，检查 UTF-8 声明、响应式 `viewport`、核心交互 ID 与 JS `fetch()` 调用。

In [2]:
result = validate_frontend_file()
print("页面标题:", result.title)
print("UTF-8 编码声明:", result.has_utf8_meta)
print("Viewport 响应式声明:", result.has_viewport_meta)
print("JS fetch() 调用:", result.has_fetch_call)
print("检出的 DOM ID:", result.found_ids)
print("校验是否通过:", result.is_valid)
assert result.is_valid, f"前端校验失败: {result.errors}"

页面标题: 智枢企业级智能体对话台
UTF-8 编码声明: True
Viewport 响应式声明: True
JS fetch() 调用: True
检出的 DOM ID: ['api-status', 'clear-btn', 'chat-messages', 'chat-form', 'user-input', 'send-btn']
校验是否通过: True


## 3. 检查 JS `fetch()` 通信逻辑与 Flexbox 气泡样式
验证页面中包含标准 `POST` JSON 请求头与用户/助手双侧气泡样式。

In [3]:
html_text = get_index_html_path().read_text(encoding="utf-8")
checks = {
    "总行数": len(html_text.splitlines()),
    "包含用户气泡样式 (.message.user)": ".message.user" in html_text,
    "包含助手气泡样式 (.message.assistant)": ".message.assistant" in html_text,
    "包含响应式媒体查询 (@media)": "@media" in html_text,
    "包含 POST 请求方法": "POST" in html_text,
    "包含 application/json 请求头": "application/json" in html_text,
}
print(json.dumps(checks, ensure_ascii=False, indent=2))
assert all(v is True for k, v in checks.items() if k != "总行数")

{
  "总行数": 465,
  "包含用户气泡样式 (.message.user)": true,
  "包含助手气泡样式 (.message.assistant)": true,
  "包含响应式媒体查询 (@media)": true,
  "包含 POST 请求方法": true,
  "包含 application/json 请求头": true
}


## 4. 负向校验演示与静态资源托管信息
验证 `validate_html_content()` 能够准确拦截缺失 DOM ID 或缺少 `fetch()` 的异常页面，并输出本地静态预览信息。

In [4]:
broken_html = "<html><head><title>Broken</title></head><body><div id=\"chat-messages\"></div></body></html>"
broken_res = validate_html_content(broken_html)
print("异常 HTML 校验结果:", broken_res.is_valid)
print("检出错误清单:", broken_res.errors)
assert not broken_res.is_valid

serving_info = describe_static_serving(port=8080)
print("\n静态预览配置:")
print(json.dumps(serving_info, ensure_ascii=False, indent=2))

异常 HTML 校验结果: False
检出错误清单: ['缺少 <meta charset="UTF-8"> 声明', '缺少响应式 <meta name="viewport"> 声明', '缺少必要 DOM ID: chat-form, user-input, send-btn', '脚本中未检测到 JS fetch() API 调用']

静态预览配置:
{
  "frontend_dir": "/home/antigravity/ab6539763/-AI-Agent/project_iterations/day_22/frontend",
  "index_html": "/home/antigravity/ab6539763/-AI-Agent/project_iterations/day_22/frontend/index.html",
  "preview_url": "http://127.0.0.1:8080/index.html",
  "serve_command": "python -m http.server 8080 --directory /home/antigravity/ab6539763/-AI-Agent/project_iterations/day_22/frontend"
}
